# bronze2silver_dq_nbk
Runs **after** `bronze2silver_transform_nbk` has loaded data into the silver table.
Reads DQ rules from `elt_dq_config`, queries the silver table, evaluates each rule,
and writes a result row per rule into `elt_dq_results`.

Supported rule types: `allowed_values` | `not_null` | `range` | `regex`

Actions: `warn` = log only | `flag` = not yet implemented (post-load flagging requires ALTER TABLE)

In [ ]:
# Cell 1 — Parameters (Fabric injects values at runtime)
table_id              = ""
tablename             = ""
silver_schema         = "silver"
pipeline_run_id       = ""
pipeline_trigger_time = ""

warehouse_endpoint    = "nv3l45p7irxunojdrwsud5otge-isco3hrff5zuzpeqnjwvvo6u5i.datawarehouse.fabric.microsoft.com"
warehouse_name        = "MDF_WAREHOUSE_DEV_001"

In [ ]:
# Cell 2 — Imports and warehouse connection
import re, struct, json
import pandas as pd
import pyodbc
from azure.identity import DefaultAzureCredential


def _connect_warehouse() -> pyodbc.Connection:
    token = DefaultAzureCredential().get_token('https://database.windows.net/.default').token
    token_bytes = bytes(token, 'utf-8')
    exp_token = b''
    for b in token_bytes:
        exp_token += bytes({b}) + bytes(1)
    attrs = {1256: struct.pack('=i', len(exp_token)) + exp_token}
    conn_str = (
        'Driver={ODBC Driver 18 for SQL Server};'
        f'Server={warehouse_endpoint},1433;'
        f'Database={warehouse_name};'
        'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
    )
    return pyodbc.connect(conn_str, attrs_before=attrs, autocommit=False)


conn = _connect_warehouse()
print('warehouse connected')

In [ ]:
# Cell 3 — Fetch active DQ rules for this table_id
silver_fqn = f"[{silver_schema}].[{tablename}]"

with conn.cursor() as cur:
    cur.execute(
        "SELECT rule_id, column_name, rule_type, rule_value, action "
        "FROM mdf_platform_orchestration.elt_dq_config "
        "WHERE table_id = ? AND is_active = 1",
        (int(table_id),)
    )
    rules = cur.fetchall()

print(f"DQ rules loaded: {len(rules)} for {silver_fqn}")

if not rules:
    print("No active DQ rules — notebook complete")
    dbutils.notebook.exit(json.dumps({'status': 'NO_RULES', 'table': tablename}))

In [ ]:
# Cell 4 — Load current silver rows into pandas (is_current = 1 only)
with conn.cursor() as cur:
    cur.execute(f"SELECT * FROM {silver_fqn} WHERE is_current = 1")
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()

df_silver = pd.DataFrame.from_records(rows, columns=cols)
total = len(df_silver)
print(f"silver rows loaded: {total:,}")

In [ ]:
# Cell 5 — Evaluate each rule and collect results
results = []

for rule_id, col_name, rule_type, rule_value, action in rules:
    fail_count = 0

    if col_name not in df_silver.columns:
        print(f"  SKIP rule {rule_id}: column '{col_name}' not in silver table")
        continue

    series = df_silver[col_name]

    if rule_type == 'not_null':
        fail_count = int(series.isna().sum())

    elif rule_type == 'allowed_values':
        allowed = {v.strip() for v in rule_value.split(',')}
        fail_count = int((~series.astype(str).isin(allowed) & series.notna()).sum())

    elif rule_type == 'range':
        params = dict(p.split(':') for p in rule_value.split(','))
        numeric = pd.to_numeric(series, errors='coerce')
        mask = pd.Series([False] * total)
        if 'min' in params:
            mask |= numeric < float(params['min'])
        if 'max' in params:
            mask |= numeric > float(params['max'])
        fail_count = int(mask.sum())

    elif rule_type == 'regex':
        pattern = re.compile(rule_value)
        fail_count = int(series.astype(str).apply(lambda v: not pattern.fullmatch(v)).sum())

    pass_rate = round(100 * (total - fail_count) / total, 2) if total else None
    status = 'PASS' if fail_count == 0 else f'FAIL ({fail_count} rows)'
    print(f"  rule {rule_id} [{rule_type}] {col_name}: {status}  pass_rate={pass_rate}%")

    results.append((pipeline_run_id, int(table_id), tablename, col_name,
                    rule_type, rule_value, action, fail_count, total, pass_rate))

print(f"\nrules evaluated: {len(results)}")

In [ ]:
# Cell 6 — Write results to elt_dq_results
insert_sql = (
    "INSERT INTO mdf_platform_orchestration.elt_dq_results "
    "(pipeline_run_id, table_id, tablename, column_name, rule_type, rule_value, "
    " action, fail_count, total_count, pass_rate_pct) "
    "VALUES (?,?,?,?,?,?,?,?,?,?)"
)
with conn.cursor() as cur:
    cur.executemany(insert_sql, results)
conn.commit()
conn.close()

summary = {
    'status':        'SUCCESS',
    'table':         tablename,
    'rules_checked': len(results),
    'failures':      sum(1 for r in results if r[7] > 0),
    'run_id':        pipeline_run_id,
}
print(json.dumps(summary, indent=2))